# 02 · Demand Forecasting and Out-of-Year Evaluation

**A transparent, reproducible CatBoost benchmark for BIXI station demand.**

Goal: forecast the number of arrivals and departures at each station during a **15-minute interval**.
Models are trained on BIXI trip records from **2022–2024**, then tested without retraining on
**2025 and 2026**, using only historical station identities and comparable station-calendar-days.

**Primary result:** the original experiments reported meaningful MAE reductions for arrival and
departure forecasts. We regenerate those metrics from local data rather than hard-code them as
notebook outputs. Results are only comparable to the original experiment if the upstream station
matching, active-day grid, model versions, and data cuts agree.



## 1. Inputs, dependencies and reproducibility

 Training datasets from `01_Data_Preparation.ipynb`  :

- `data/processed/bixi_2022_complete.parquet`
- `data/processed/bixi_2023_complete.parquet`
- `data/processed/bixi_2024_complete.parquet`

The 2025–2026 test grids at:

- `data/processed/bixi_2025_complete.parquet`
- `data/processed/bixi_2026_complete.parquet`



In [ ]:
from pathlib import Path
import time
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd(), Path.cwd().parent]
                     if (p / "data" / "processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from your BIXI project directory; data/processed is missing.")

PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)
TRAIN_YEARS = [2022, 2023, 2024]
TEST_YEARS = [2025, 2026]
ALL_FILES = {year: PROCESSED / f"bixi_{year}_complete.parquet"
             for year in TRAIN_YEARS + TEST_YEARS}
missing = [str(path) for path in ALL_FILES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing processed data: {missing}")

FEATURES = ["station", "year", "month", "day_of_week", "hour", "time_of_day_slot"]
con = duckdb.connect()
print("Training years:", TRAIN_YEARS, "| Test years:", TEST_YEARS)

## 2. Feature engineering and training sample

 the six predictors are : station identity, year, month, day of week,
hour, and quarter-hour slot. Calendar features are computed in Montréal time after reading the
UTC `time_slot`. Training keeps April–October only

In [ ]:
FEATURE_SQL = """
SELECT
    station,
    time_slot,
    EXTRACT(YEAR FROM local_time)::INTEGER AS year,
    EXTRACT(MONTH FROM local_time)::INTEGER AS month,
    EXTRACT(DOW FROM local_time)::INTEGER AS day_of_week,
    EXTRACT(HOUR FROM local_time)::INTEGER AS hour,
    (EXTRACT(HOUR FROM local_time) * 4
      + FLOOR(EXTRACT(MINUTE FROM local_time) / 15))::INTEGER AS time_of_day_slot,
    arrivals, departures
FROM (
    SELECT station, time_slot, arrivals, departures,
           timezone('America/Toronto', to_timestamp(time_slot * 900)) AS local_time
    FROM read_parquet(?)
)
"""

training_paths = [str(ALL_FILES[year]) for year in TRAIN_YEARS]
train = con.execute("""
    WITH features AS (""" + FEATURE_SQL + """)
    SELECT station, year, month, day_of_week, hour, time_of_day_slot,
           arrivals, departures
    FROM features
    WHERE month BETWEEN 4 AND 10
      AND hash(station, time_slot) % 100 < 4
""", [training_paths]).df()

assert len(train) > 0 and train[FEATURES].notna().all().all()
assert train["year"].isin(TRAIN_YEARS).all()
train["station"] = train["station"].astype(str)
display(train.groupby("year").agg(
    intervals=("station", "size"), stations=("station", "nunique"),
    arrivals_mean=("arrivals", "mean"), departures_mean=("departures", "mean")
))

## 3. Train two CatBoost Poisson regressors

The model predicts **non negative expected counts**. Poisson loss is suitable as an initial
count model; it does not guarantee that actual station arrivals/departures follow independent
Poisson distributions. Hyperparameters below reproduce the final-model training notebook.

In [ ]:
parameters = dict(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Poisson",
    random_seed=42,
    thread_count=4,
    verbose=100,
    allow_writing_files=False,
)

arrival_model = CatBoostRegressor(**parameters)
departure_model = CatBoostRegressor(**parameters)
X_train = train[FEATURES]
start = time.perf_counter()
arrival_model.fit(X_train, train["arrivals"], cat_features=["station"])
departure_model.fit(X_train, train["departures"], cat_features=["station"])
print(f"Training completed in {time.perf_counter() - start:.1f} seconds")

arrival_model.save_model(str(MODELS / "bixi_arrivals_2022_2024.cbm"))
departure_model.save_model(str(MODELS / "bixi_departures_2022_2024.cbm"))

## 4. Construct a matched 2025–2026 evaluation 

Station names and operating periods change from year to year. 

In [ ]:
test_files = {year: str(ALL_FILES[year]) for year in TEST_YEARS}

station_days = {}
for year in TEST_YEARS:
    station_days[year] = con.execute("""
        WITH activity AS (
            SELECT station,
                timezone('America/Toronto', to_timestamp(time_slot * 900)) AS local_time
            FROM read_parquet(?)
        )
        SELECT DISTINCT station,
            EXTRACT(MONTH FROM local_time)::INTEGER AS month,
            EXTRACT(DAY FROM local_time)::INTEGER AS day
        FROM activity
        WHERE local_time >= MAKE_TIMESTAMP(?, 4, 1, 0, 0, 0)
          AND local_time <= MAKE_TIMESTAMP(?, 9, 26, 16, 15, 0)
    """, [test_files[year], year, year]).df()

paired_days = station_days[2025].merge(station_days[2026],
                                       on=["station", "month", "day"], how="inner")
assert not paired_days.duplicated(["station", "month", "day"]).any()
assert len(paired_days) > 0
print("Paired station-days:", len(paired_days))
print("Stations represented:", paired_days["station"].nunique())
con.register("paired_days", paired_days)

## 5. Evaluate in batches 

MAE is computed over all matched station-intervals. We compare arrival, departure, and
**net-flow** estimates with a simple predict-zero baseline. Net-flow is important because
inventory changes depend on `arrivals − departures`

In [ ]:
EVAL_SQL = """
WITH base AS (
    SELECT a.station, a.time_slot, a.arrivals, a.departures,
           timezone('America/Toronto', to_timestamp(a.time_slot * 900)) AS local_time
    FROM read_parquet(?) a
), selected AS (
    SELECT b.* FROM base b
    INNER JOIN paired_days d
      ON b.station = d.station
     AND EXTRACT(MONTH FROM b.local_time)::INTEGER = d.month
     AND EXTRACT(DAY FROM b.local_time)::INTEGER = d.day
    WHERE b.local_time >= MAKE_TIMESTAMP(?, 4, 1, 0, 0, 0)
      AND b.local_time <= MAKE_TIMESTAMP(?, 9, 26, 16, 15, 0)
)
SELECT station,
    EXTRACT(YEAR FROM local_time)::INTEGER AS year,
    EXTRACT(MONTH FROM local_time)::INTEGER AS month,
    EXTRACT(DOW FROM local_time)::INTEGER AS day_of_week,
    EXTRACT(HOUR FROM local_time)::INTEGER AS hour,
    (EXTRACT(HOUR FROM local_time) * 4
      + FLOOR(EXTRACT(MINUTE FROM local_time) / 15))::INTEGER AS time_of_day_slot,
    arrivals, departures
FROM selected
"""


def evaluate_year(year, batch_size=100_000):
    cursor = con.execute(EVAL_SQL, [test_files[year], year, year])
    names = [item[0] for item in cursor.description]
    sums = {name: 0.0 for name in [
        "arrival_error", "departure_error", "net_error",
        "zero_arrival", "zero_departure", "zero_net",
        "actual_arrivals", "actual_departures", "pred_arrivals", "pred_departures"
    ]}
    count = 0
    while True:
        rows = cursor.fetchmany(batch_size)
        if not rows:
            break
        batch = pd.DataFrame(rows, columns=names)
        batch["station"] = batch["station"].astype(str)
        a = batch["arrivals"].to_numpy(dtype=float)
        d = batch["departures"].to_numpy(dtype=float)
        pred_a = np.maximum(0, arrival_model.predict(batch[FEATURES]))
        pred_d = np.maximum(0, departure_model.predict(batch[FEATURES]))

        count += len(batch)
        sums["arrival_error"] += np.abs(a - pred_a).sum()
        sums["departure_error"] += np.abs(d - pred_d).sum()
        sums["net_error"] += np.abs((a - d) - (pred_a - pred_d)).sum()
        sums["zero_arrival"] += np.abs(a).sum()
        sums["zero_departure"] += np.abs(d).sum()
        sums["zero_net"] += np.abs(a - d).sum()
        sums["actual_arrivals"] += a.sum()
        sums["actual_departures"] += d.sum()
        sums["pred_arrivals"] += pred_a.sum()
        sums["pred_departures"] += pred_d.sum()

    if count == 0:
        raise ValueError(f"No matched test intervals found for {year}.")
    return {"n": count, **sums}


results = {}
for year in TEST_YEARS:
    start = time.perf_counter()
    results[year] = evaluate_year(year)
    print(f"{year}: {results[year]['n']:,} intervals, {time.perf_counter()-start:.1f} seconds")

## 6. Metrics and visualization

**Lower MAE is better.** Positive improvement indicates lower error than predicting zero.
The plot shows the main arrival/departure counts, 

In [ ]:
rows = []
for year, r in results.items():
    for target, model_key, baseline_key in [
        ("Arrivals", "arrival_error", "zero_arrival"),
        ("Departures", "departure_error", "zero_departure"),
        ("Net flow", "net_error", "zero_net"),
    ]:
        mae = r[model_key] / r["n"]
        baseline = r[baseline_key] / r["n"]
        improvement = 100 * (baseline - mae) / baseline if baseline else np.nan
        rows.append(dict(year=year, target=target, intervals=r["n"],
                         model_mae=mae, zero_baseline_mae=baseline,
                         improvement_pct=improvement))

metrics = pd.DataFrame(rows)
display(metrics.round(4))

main = metrics[metrics.target.isin(["Arrivals", "Departures"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, year in zip(axes, TEST_YEARS):
    subset = main[main.year == year]
    x = np.arange(len(subset))
    ax.bar(x - .2, subset.zero_baseline_mae, .4, label="Predict zero", color="#AAB7C4")
    ax.bar(x + .2, subset.model_mae, .4, label="CatBoost", color="#2B956A")
    ax.set_xticks(x, subset.target)
    ax.set_title(f"{year}: unseen-year evaluation")
    ax.set_ylabel("MAE (bikes / 15-minute interval)")
    ax.grid(axis="y", alpha=.18)
axes[0].legend(frameon=False)
fig.suptitle("BIXI demand forecasting: error versus a zero-count baseline")
fig.tight_layout()
plt.show()

In [ ]:
bias_rows = []
for year, r in results.items():
    for label, akey, pkey in [
        ("Arrivals", "actual_arrivals", "pred_arrivals"),
        ("Departures", "actual_departures", "pred_departures"),
    ]:
        actual_mean, predicted_mean = r[akey] / r["n"], r[pkey] / r["n"]
        bias_rows.append(dict(year=year, target=label, actual_mean=actual_mean,
                              predicted_mean=predicted_mean,
                              bias=predicted_mean - actual_mean))
display(pd.DataFrame(bias_rows).round(4))